# LatentSync · 高清唇形同步数字人（Colab GPU）

**当前 SOTA 视频驱动唇形同步**（字节开源，效果远超 Wav2Lip，接近旗博士）。
输入：**一段真人正脸说话视频** + 口播音频 → 输出：同一个人、同样头动，但嘴形对上新口播的高清视频。

**先做**：`代码执行程序 → 更改运行时类型 → T4 GPU`，然后逐格运行。

In [ ]:
# 1. 确认 GPU
!nvidia-smi -L

In [ ]:
# 2. clone LatentSync + 装依赖（约 5-10 分钟）
%cd /content
import os
if not os.path.isdir('LatentSync'):
    !git clone -q https://github.com/bytedance/LatentSync.git
%cd LatentSync
!apt-get -qq -y install libgl1 > /dev/null 2>&1
!pip install -q -r requirements.txt
print("依赖安装完成")

In [ ]:
# 3. 下载模型权重（HuggingFace 镜像；latentsync_unet ~5GB，几分钟）
%cd /content/LatentSync
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
!huggingface-cli download ByteDance/LatentSync-1.6 whisper/tiny.pt --local-dir checkpoints
!huggingface-cli download ByteDance/LatentSync-1.6 latentsync_unet.pt --local-dir checkpoints
!ls -la checkpoints

In [ ]:
# 4. 上传你的【真人正脸说话视频】+ 生成口播音频
%cd /content/LatentSync
from google.colab import files
print(">>> 请选择你的驱动视频（mp4，正脸单人）：")
up = files.upload()
import shutil
vname = list(up.keys())[0]
shutil.move(vname, "input_video.mp4")
print("视频已保存: input_video.mp4")

# 口播文案 -> edge-tts 音频（想用自己的音频就上传 audio.wav 覆盖）
!pip install -q edge-tts
TEXT = "大家好，欢迎观看这条由数字人生成的口播视频，效果测试。"
VOICE = "zh-CN-XiaoxiaoNeural"
import asyncio, edge_tts
asyncio.run(edge_tts.Communicate(TEXT, VOICE).save("audio.mp3"))
!ffmpeg -y -i audio.mp3 -ar 16000 audio.wav -loglevel error
print("音频就绪: audio.wav")

In [ ]:
# 5. 跑 LatentSync（stage2_512 高清；T4 上约几分钟）
%cd /content/LatentSync
!python -m scripts.inference \
  --unet_config_path "configs/unet/stage2_512.yaml" \
  --inference_ckpt_path "checkpoints/latentsync_unet.pt" \
  --inference_steps 20 --guidance_scale 1.5 --enable_deepcache \
  --video_path "input_video.mp4" \
  --audio_path "audio.wav" \
  --video_out_path "video_out.mp4"
# 若显存不足(OOM)：把 stage2_512.yaml 改成 stage2.yaml（256分辨率）再跑

In [ ]:
# 6. 预览 + 下载成品
from base64 import b64encode
from IPython.display import HTML
from google.colab import files
mp4 = "/content/LatentSync/video_out.mp4"
files.download(mp4)
HTML(f'<video width=360 controls src="data:video/mp4;base64,{b64encode(open(mp4,"rb").read()).decode()}"></video>')

---
- **显存不足 OOM**：第 5 格把 `stage2_512.yaml` 换成 `stage2.yaml`（256 分辨率，省显存）。
- **换声音**：想"像某个人"就上传 `audio.wav` 覆盖，或接 CosyVoice 克隆音。
- **效果不够**：`--inference_steps` 调大（如 30）更精细但更慢。